# Cohort-Level Benchmarks

This notebook reproduces the cohort-level numbers presented in the release/blog from checked-in artifacts: Gaia strict-44 ORR, Atlas fixed-k historical ORR prior, and DepMap lineage-sensitivity baseline.

It also regenerates cohort figures under `artifacts/notebook_figures/`.

In [ ]:
from pathlib import Path
import csv
import html
import json
import math
import sys

try:
    from IPython.display import SVG, Markdown, display
except Exception:
    SVG = Markdown = None
    def display(value):
        print(value)

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Run this notebook from inside the open-benchmarks repo")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from spatial_benchmarks.metrics import finite_float
from spatial_benchmarks.reproduce import (
    reproduce_atlas_orr_predictions,
    reproduce_depmap_orr_features,
    reproduce_gaia_cohort_orr,
    raise_if_drift,
)

FIG_DIR = ROOT / "artifacts/notebook_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def read_rows(path):
    with Path(path).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

def fmt(value, digits=3):
    value = finite_float(value)
    return "NA" if not math.isfinite(value) else f"{value:.{digits}f}"

def show_svg(path):
    if SVG is None:
        print(path)
    else:
        display(SVG(filename=str(path)))

def write_svg(path, body, width=760, height=520):
    svg = f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">{body}</svg>'
    path.write_text(svg + "\n", encoding="utf-8")
    return path

## Recompute Metrics

In [ ]:
GAIA = ROOT / "cohort-level-bench/model_scores/gaia"
BASELINE = ROOT / "cohort-level-bench/baseline/results"

gaia_report = reproduce_gaia_cohort_orr(
    scores_csv=GAIA / "gaia_44_strict_orr_model_scores.csv",
    metrics_csv=GAIA / "gaia_metrics.csv",
    by_disease_csv=GAIA / "gaia_by_disease_metrics.csv",
)
atlas_report = reproduce_atlas_orr_predictions(
    predictions_csv=BASELINE / "atlas_orr_predictions.csv",
    metrics_csv=BASELINE / "atlas_orr_metrics.csv",
)
depmap_report = reproduce_depmap_orr_features(
    features_csv=BASELINE / "depmap_orr_features.csv",
    metrics_csv=BASELINE / "depmap_orr_metrics.csv",
)

for report in (gaia_report, atlas_report, depmap_report):
    raise_if_drift(report)

cohort_headlines = {
    "Gaia strict ORR rows": gaia_report["metrics"]["n_rows"],
    "Gaia Pearson r": round(gaia_report["metrics"]["pearson"], 3),
    "Gaia Spearman rho": round(gaia_report["metrics"]["spearman"], 3),
    "Gaia MAE pp": round(gaia_report["metrics"]["mae_orr_pct"], 1),
    "Gaia AUC above disease median": round(gaia_report["metrics"]["auc_above_disease_median"], 3),
    "Atlas fixed-k Pearson r": round(atlas_report["metrics"]["pearson"], 3),
    "Atlas fixed-k Spearman rho": round(atlas_report["metrics"]["spearman"], 3),
    "Atlas fixed-k MAE pp": round(atlas_report["metrics"]["mae_orr_pct"], 1),
    "DepMap covered rows": depmap_report["primary_metrics"]["n_rows"],
    "DepMap Pearson r": round(depmap_report["primary_metrics"]["pearson"], 3),
    "DepMap Spearman rho": round(depmap_report["primary_metrics"]["spearman"], 3),
}
cohort_headlines

## Figure Helpers

In [ ]:
PALETTE = {
    "breast": "#2f6f9f",
    "crc": "#8d5a2b",
    "nsclc": "#65743a",
    "ovarian": "#8c4f7d",
    "pdac": "#b84a3a",
}

def scatter_observed_vs_predicted(rows, score_col, title, output_name):
    width, height = 760, 560
    left, right, top, bottom = 88, 32, 62, 82
    plot_w = width - left - right
    plot_h = height - top - bottom
    points = []
    max_value = 0.0
    for row in rows:
        x = finite_float(row.get("observed_orr_pct") or row.get("orr_pct"))
        y = finite_float(row.get(score_col))
        if math.isfinite(x) and math.isfinite(y):
            points.append((x, y, str(row.get("cohort") or row.get("disease")), row))
            max_value = max(max_value, x, y)
    axis_max = max(50, math.ceil(max_value / 10) * 10)
    def sx(value):
        return left + plot_w * value / axis_max
    def sy(value):
        return top + plot_h * (1 - value / axis_max)
    body = [
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{left}" y="32" font-family="Inter, Arial" font-size="22" font-weight="700" fill="#202124">{html.escape(title)}</text>',
        f'<line x1="{left}" y1="{top + plot_h}" x2="{left + plot_w}" y2="{top + plot_h}" stroke="#333"/>',
        f'<line x1="{left}" y1="{top}" x2="{left}" y2="{top + plot_h}" stroke="#333"/>',
        f'<line x1="{sx(0)}" y1="{sy(0)}" x2="{sx(axis_max)}" y2="{sy(axis_max)}" stroke="#9aa0a6" stroke-dasharray="5 5"/>',
    ]
    for tick in range(0, int(axis_max) + 1, 10):
        x = sx(tick)
        y = sy(tick)
        body.append(f'<line x1="{x:.1f}" y1="{top}" x2="{x:.1f}" y2="{top + plot_h}" stroke="#eef0f2"/>')
        body.append(f'<line x1="{left}" y1="{y:.1f}" x2="{left + plot_w}" y2="{y:.1f}" stroke="#eef0f2"/>')
        body.append(f'<text x="{x:.1f}" y="{top + plot_h + 24}" text-anchor="middle" font-family="Inter, Arial" font-size="12" fill="#5f6368">{tick}</text>')
        body.append(f'<text x="{left - 12}" y="{y + 4:.1f}" text-anchor="end" font-family="Inter, Arial" font-size="12" fill="#5f6368">{tick}</text>')
    for x, y, disease, row in points:
        color = PALETTE.get(disease, "#5f6368")
        label = f'{row.get("drug", "")} {disease}: observed {x:.1f}, predicted {y:.1f}'
        body.append(f'<circle cx="{sx(x):.1f}" cy="{sy(y):.1f}" r="6" fill="{color}" fill-opacity="0.86"><title>{html.escape(label)}</title></circle>')
    body.append(f'<text x="{left + plot_w / 2}" y="{height - 26}" text-anchor="middle" font-family="Inter, Arial" font-size="14" fill="#333">Observed ORR (%)</text>')
    body.append(f'<text transform="translate(24 {top + plot_h / 2}) rotate(-90)" text-anchor="middle" font-family="Inter, Arial" font-size="14" fill="#333">Predicted ORR (%)</text>')
    legend_x = left + plot_w - 150
    for i, disease in enumerate(PALETTE):
        y = top + 18 + i * 20
        body.append(f'<circle cx="{legend_x}" cy="{y}" r="5" fill="{PALETTE[disease]}"/>')
        body.append(f'<text x="{legend_x + 12}" y="{y + 4}" font-family="Inter, Arial" font-size="12" fill="#333">{disease}</text>')
    return write_svg(FIG_DIR / output_name, "".join(body), width, height)

def metric_comparison_bar(methods, output_name):
    width, height = 760, 500
    left, right, top, bottom = 82, 28, 58, 78
    plot_w = width - left - right
    plot_h = height - top - bottom
    y_min, y_max = -0.10, 0.80
    metrics = [("pearson", "Pearson r", "#2f6f9f"), ("spearman", "Spearman rho", "#b84a3a"), ("auc", "AUC", "#65743a")]
    def sy(value):
        return top + plot_h * (1 - (value - y_min) / (y_max - y_min))
    body = [
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{left}" y="32" font-family="Inter, Arial" font-size="22" font-weight="700" fill="#202124">Cohort Metric Comparison</text>',
        f'<line x1="{left}" y1="{sy(0):.1f}" x2="{left + plot_w}" y2="{sy(0):.1f}" stroke="#333"/>',
        f'<line x1="{left}" y1="{top}" x2="{left}" y2="{top + plot_h}" stroke="#333"/>',
    ]
    for tick in [-0.1, 0.0, 0.2, 0.4, 0.6, 0.8]:
        y = sy(tick)
        body.append(f'<line x1="{left}" y1="{y:.1f}" x2="{left + plot_w}" y2="{y:.1f}" stroke="#eef0f2"/>')
        body.append(f'<text x="{left - 10}" y="{y + 4:.1f}" text-anchor="end" font-family="Inter, Arial" font-size="12" fill="#5f6368">{tick:.1f}</text>')
    group_w = plot_w / len(methods)
    bar_w = group_w / 5
    for i, method in enumerate(methods):
        center = left + group_w * (i + 0.5)
        for j, (key, _, color) in enumerate(metrics):
            value = finite_float(method[key])
            x = center + (j - 1) * bar_w * 1.25 - bar_w / 2
            y0 = sy(0)
            y1 = sy(value)
            h = abs(y0 - y1)
            y = min(y0, y1)
            body.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_w:.1f}" height="{h:.1f}" fill="{color}"/>')
            body.append(f'<text x="{x + bar_w/2:.1f}" y="{y - 5:.1f}" text-anchor="middle" font-family="Inter, Arial" font-size="11" fill="#333">{value:.2f}</text>')
        body.append(f'<text x="{center:.1f}" y="{height - 34}" text-anchor="middle" font-family="Inter, Arial" font-size="13" fill="#333">{html.escape(method["label"])}</text>')
    for j, (_, label, color) in enumerate(metrics):
        x = left + j * 150
        body.append(f'<rect x="{x}" y="{height - 18}" width="12" height="12" fill="{color}"/>')
        body.append(f'<text x="{x + 18}" y="{height - 8}" font-family="Inter, Arial" font-size="12" fill="#333">{label}</text>')
    return write_svg(FIG_DIR / output_name, "".join(body), width, height)

## Generate Cohort Figures

In [ ]:
gaia_rows = read_rows(GAIA / "gaia_44_strict_orr_model_scores.csv")
atlas_rows = read_rows(BASELINE / "atlas_orr_predictions.csv")

fig1 = scatter_observed_vs_predicted(
    gaia_rows,
    "gaia_predicted_orr_pct",
    "Strict 44 Cohorts: Observed vs Gaia Predicted ORR",
    "cohort_gaia_observed_vs_predicted.svg",
)
fig2 = scatter_observed_vs_predicted(
    atlas_rows,
    "atlas_mono_disease_therapy_shrink_k8",
    "Strict 44 Cohorts: Observed vs Atlas Prior ORR",
    "cohort_atlas_observed_vs_predicted.svg",
)
fig3 = metric_comparison_bar(
    [
        {"label": "Gaia", "pearson": gaia_report["metrics"]["pearson"], "spearman": gaia_report["metrics"]["spearman"], "auc": gaia_report["metrics"]["auc_above_disease_median"]},
        {"label": "Atlas", "pearson": atlas_report["metrics"]["pearson"], "spearman": atlas_report["metrics"]["spearman"], "auc": atlas_report["metrics"]["auc_above_disease_median"]},
        {"label": "DepMap", "pearson": depmap_report["primary_metrics"]["pearson"], "spearman": depmap_report["primary_metrics"]["spearman"], "auc": depmap_report["primary_metrics"]["auc_above_disease_median"]},
    ],
    "cohort_metric_comparison.svg",
)

for fig in (fig1, fig2, fig3):
    show_svg(fig)

[str(fig.relative_to(ROOT)) for fig in (fig1, fig2, fig3)]